# 1. Imports 

In [ ]:
import os
import pandas as pd
import numpy as np
import pm4py
import re
from ProcessMiningAgent_CodeGeneration_V4_o4 import ProcessMiningAgent

# 2. Directories

In [ ]:
base_dir = r"C:\Users\carla\OneDrive\Ambiente de Trabalho\Tese\MeuChatBot"
output_root = os.path.join(base_dir, "Results")
os.makedirs(output_root, exist_ok=True)

In [ ]:
questions_df = pd.read_excel(
    os.path.join(base_dir, "teste-o-meu-chat-bot_teste.xlsx"),
    sheet_name="Sheet1" 
)

# 3. API Keys

In [ ]:
#Put your PAI Keys here

# 4.Datasets

In [ ]:
datasets = {
    "sepsis": {
        "name": "Sepsis Cases",
        "log_path": r"C:\Users\carla\OneDrive\Ambiente de Trabalho\Tese\MeuChatBot\Data\Sepsis Cases - Event Log_1_all\Sepsis Cases - Event Log.xes\Sepsis Cases - Event Log.xes",
        "questions_file": os.path.join(base_dir, "teste-o-meu-chat-bot_teste.xlsx"),
        "description": "Log de eventos de casos de sepsis em hospital"
    },
    "EletronicInvoice": {
        "name": "Eletronic Invoice Cases",
        "log_path": r"C:\Users\carla\OneDrive\Ambiente de Trabalho\Tese\MeuChatBot\Data\Logs_Caldeira\Electronic Invoicing Event Logs_1_all\ElectronicInvoicingENG.xes",
        "questions_file": os.path.join(base_dir, "teste-o-meu-chat-bot_teste.xlsx"),
        "description": "Log de eventos de faturas eletrónicas"
    },
    "RequestForPayment": {
        "name": "Request For Payment Cases",
        "log_path": r"C:\Users\carla\OneDrive\Ambiente de Trabalho\Tese\MeuChatBot\Data\BPI Challenge 2020_ Request For Payment_1_all (1)\RequestForPayment.xes\RequestForPayment.xes",
        "questions_file": os.path.join(base_dir, "teste-o-meu-chat-bot_teste.xlsx"),
        "description": "Log de eventos de pedidos de pagamento"
    }
}

In [ ]:
for key, dataset in datasets.items():
    print(f"Carregando dataset {dataset['name']}...")
    
    log = pm4py.read_xes(dataset['log_path'])
    df = pm4py.convert_to_dataframe(log)
    
    questions = pd.read_excel(dataset['questions_file'])
    
    datasets[key]['log'] = log
    datasets[key]['df'] = df
    datasets[key]['questions'] = questions
    
    print(f"  - {len(df)} eventos carregados")
    print(f"  - {len(questions)} perguntas disponíveis")

In [ ]:
sepsis_log = datasets["sepsis"]["log"]
sepsis_df = datasets["sepsis"]["df"]
sepsis_questions = datasets["sepsis"]["questions"]

eletronic_invoice_log = datasets["EletronicInvoice"]["log"]
eletronic_invoice_df = datasets["EletronicInvoice"]["df"]
eletronic_invoice_questions = datasets["EletronicInvoice"]["questions"]

request_for_payment_log = datasets["RequestForPayment"]["log"]
request_for_payment_df = datasets["RequestForPayment"]["df"]
request_for_payment_questions = datasets["RequestForPayment"]["questions"]

# 5. Prompts

## 5.1 Code Generation

### 5.1.1 Prompt

In [ ]:
GENERATION_PROMPTS = {
"ZERO_SHOT": """You are a process mining assistant that analyzes event logs and generates Python code to create a visualization in response to a process analysis question.

The log is loaded in the variable `log` (PM4PY EventLog object), and a corresponding pandas DataFrame is available as `df`. 

1. **Relevant Columns:** Specify which columns are necessary to answer the question and why (e.g., `concept:name`, `case:concept:name`, `time:timestamp`, `org:group`, etc.).

Your task is to return only executable Python code that produces the appropriate plot to answer the following question. Do not include explanations or comments.

Question:
"{question}"
"""
,

"ONE_SHOT": """You are a process mining assistant that analyzes event logs and generates Python code to create a visualization in response to a process analysis question.

The log is loaded in the variable `log` (PM4PY EventLog object), and a corresponding pandas DataFrame is available as `df`. 

1. **Relevant Columns:** Specify which columns are necessary to answer the question and why (e.g., `concept:name`, `case:concept:name`, `time:timestamp`, `org:group`, etc.).

Your task is to return only executable Python code that produces the appropriate plot to answer the question. Do not include explanations or comments.

---

Question: Show a Directly-Follows Graph that highlights how activities are connected in the process, based on frequency.

```python
from pm4py.algo.discovery.dfg import algorithm as dfg_discovery
from pm4py.visualization.dfg import visualizer as dfg_visualization
from IPython.display import Image, display

# Generate DFG from the event log
dfg = dfg_discovery.apply(log)

# Configure visualization parameters
parameters = {{
    "format": "png",
    "bgcolor": "white",
    "edge_threshold": 0.01
}}

# Save and display the DFG visualization
gviz = dfg_visualization.apply(dfg, log=log, parameters=parameters)
gviz


Now, given the following question, generate the code:

Question: "{question}" """

,

"TWO_SHOT": """You are a process mining assistant that analyzes event logs and generates Python code to create a visualization in response to a process analysis question.

The log is loaded in the variable `log` (PM4PY EventLog object), and a corresponding pandas DataFrame is available as `df`. 

1. **Relevant Columns:** Specify which columns are necessary to answer the question and why (e.g., `concept:name`, `case:concept:name`, `time:timestamp`, `org:group`, etc.).

Your task is to return only executable Python code that produces the appropriate plot to answer the question. Do not include explanations or comments.

---

Question: Show a Directly-Follows Graph that highlights how activities are connected in the process, based on frequency.

```python
from pm4py.algo.discovery.dfg import algorithm as dfg_discovery
from pm4py.visualization.dfg import visualizer as dfg_visualization
from IPython.display import Image, display

# Generate DFG from the event log
dfg = dfg_discovery.apply(log)

# Configure visualization parameters
parameters = {{
    "format": "png",
    "bgcolor": "white",
    "edge_threshold": 0.01
}}


# Save and display the DFG visualization
gviz = dfg_visualization.apply(dfg, log=log, parameters=parameters)
gviz


---

Question: Create a histogram showing how long cases typically last in the process, with mean and standard deviation highlighted.

```python
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Convert timestamp to datetime
if not pd.api.types.is_datetime64_any_dtype(df['time:timestamp']):
    df['time:timestamp'] = pd.to_datetime(df['time:timestamp'])

# Compute durations for each case
case_durations = (
    df.groupby('case:concept:name')['time:timestamp']
    .agg(['min', 'max'])
    .assign(duration_hours=lambda x: (x['max'] - x['min']).dt.total_seconds() / 3600)
)

# Summary statistics
mean_dur = case_durations['duration_hours'].mean()
std_dur = case_durations['duration_hours'].std()
median_dur = case_durations['duration_hours'].median()

# Histogram
plt.figure(figsize=(12, 6))
sns.histplot(case_durations['duration_hours'], bins=30, kde=True, color="skyblue")
plt.axvline(mean_dur, color='red', linestyle='dashed', linewidth=2, label=f"Mean: {mean_dur:.2f}h")
plt.axvline(mean_dur + std_dur, color='orange', linestyle='dotted', linewidth=2, label=f"Mean + 1 STD: {mean_dur + std_dur:.2f}h")
plt.axvline(mean_dur - std_dur, color='orange', linestyle='dotted', linewidth=2, label=f"Mean - 1 STD: {mean_dur - std_dur:.2f}h")
plt.axvline(median_dur, color='green', linestyle='dashdot', linewidth=2, label=f"Median: {median_dur:.2f}h")
plt.title("Case Duration Distribution")
plt.xlabel("Duration (hours)")
plt.ylabel("Number of Cases")
plt.legend()
plt.tight_layout()
plt.show()

# Boxplot
plt.figure(figsize=(10, 5))
sns.boxplot(x=case_durations['duration_hours'], color="lightgrey")
plt.title("Boxplot of Case Durations")
plt.xlabel("Duration (hours)")
plt.tight_layout()
plt.show()

# Print summary stats
print(f"Mean: {mean_dur:.2f}h | Std Dev: {std_dur:.2f}h | Median: {median_dur:.2f}h")
print(f"Min: {case_durations['duration_hours'].min():.2f}h | Max: {case_durations['duration_hours'].max():.2f}h")


Now, given the following question, generate the code:

Question: "{question}" """

,

"THREE_SHOT": """You are a process mining assistant that analyzes event logs and generates Python code to create a visualization in response to a process analysis question.

The log is loaded in the variable `log` (PM4PY EventLog object), and a corresponding pandas DataFrame is available as `df`. 

1. **Relevant Columns:** Specify which columns are necessary to answer the question and why (e.g., `concept:name`, `case:concept:name`, `time:timestamp`, `org:group`, etc.).

Your task is to return only executable Python code that produces the appropriate plot to answer the question. Do not include explanations or comments.

---

Question: Show a Directly-Follows Graph that highlights how activities are connected in the process, based on frequency.

```python
from pm4py.algo.discovery.dfg import algorithm as dfg_discovery
from pm4py.visualization.dfg import visualizer as dfg_visualization
from IPython.display import Image, display

# Generate DFG from the event log
dfg = dfg_discovery.apply(log)

# Configure visualization parameters
parameters = {{
    "format": "png",
    "bgcolor": "white",
    "edge_threshold": 0.01
}}

# Save and display the DFG visualization
gviz = dfg_visualization.apply(dfg, log=log, parameters=parameters)
gviz


---

Question: Create a histogram showing how long cases typically last in the process, with mean and standard deviation highlighted.

```python
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Convert timestamp to datetime
if not pd.api.types.is_datetime64_any_dtype(df['time:timestamp']):
    df['time:timestamp'] = pd.to_datetime(df['time:timestamp'])

# Compute durations for each case
case_durations = (
    df.groupby('case:concept:name')['time:timestamp']
    .agg(['min', 'max'])
    .assign(duration_hours=lambda x: (x['max'] - x['min']).dt.total_seconds() / 3600)
)

# Summary statistics
mean_dur = case_durations['duration_hours'].mean()
std_dur = case_durations['duration_hours'].std()
median_dur = case_durations['duration_hours'].median()

# Histogram
plt.figure(figsize=(12, 6))
sns.histplot(case_durations['duration_hours'], bins=30, kde=True, color="skyblue")
plt.axvline(mean_dur, color='red', linestyle='dashed', linewidth=2, label=f"Mean: {mean_dur:.2f}h")
plt.axvline(mean_dur + std_dur, color='orange', linestyle='dotted', linewidth=2, label=f"Mean + 1 STD: {mean_dur + std_dur:.2f}h")
plt.axvline(mean_dur - std_dur, color='orange', linestyle='dotted', linewidth=2, label=f"Mean - 1 STD: {mean_dur - std_dur:.2f}h")
plt.axvline(median_dur, color='green', linestyle='dashdot', linewidth=2, label=f"Median: {median_dur:.2f}h")
plt.title("Case Duration Distribution")
plt.xlabel("Duration (hours)")
plt.ylabel("Number of Cases")
plt.legend()
plt.tight_layout()
plt.show()

# Print summary stats
print(f"Mean: {{mean_dur:.2f}}h | Std Dev: {{std_dur:.2f}}h | Median: {{median_dur:.2f}}h")
print(f"Min: {{case_durations['duration_hours'].min():.2f}}h | Max: {{case_durations['duration_hours'].max():.2f}}h")


---

Question: Generate a bar chart of activity frequencies, sorted to emphasize the most and least frequent activities

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Count activity frequencies
activity_counts = df['concept:name'].value_counts().reset_index()
activity_counts.columns = ['Activity', 'Frequency']

# Sort from most to least frequent
activity_counts = activity_counts.sort_values(by='Frequency', ascending=False)

# Create bar chart
plt.figure(figsize=(12, 6))
sns.barplot(x='Activity', y='Frequency', data=activity_counts)
plt.xticks(rotation=45, ha='right')
plt.title('Frequency of Activities')
plt.xlabel('Activity')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()"""

,

"FOUR_SHOT": """You are a process mining assistant that analyzes event logs and generates Python code to create a visualization in response to a process analysis question.

The log is loaded in the variable `log` (PM4PY EventLog object), and a corresponding pandas DataFrame is available as `df`. 

1. **Relevant Columns:** Specify which columns are necessary to answer the question and why (e.g., `concept:name`, `case:concept:name`, `time:timestamp`, `org:group`, etc.).

Your task is to return only executable Python code that produces the appropriate plot to answer the question. Do not include explanations or comments.

---

Question: Show a Directly-Follows Graph that highlights how activities are connected in the process, based on frequency.

```python
from pm4py.algo.discovery.dfg import algorithm as dfg_discovery
from pm4py.visualization.dfg import visualizer as dfg_visualization
from IPython.display import Image, display

# Generate DFG from the event log
dfg = dfg_discovery.apply(log)

# Configure visualization parameters
parameters = {{
    "format": "png",
    "bgcolor": "white",
    "edge_threshold": 0.01
}}

# Save and display the DFG visualization
gviz = dfg_visualization.apply(dfg, log=log, parameters=parameters)
gviz


---

Question: Create a histogram showing how long cases typically last in the process, with mean and standard deviation highlighted.

```python
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Convert timestamp to datetime
if not pd.api.types.is_datetime64_any_dtype(df['time:timestamp']):
    df['time:timestamp'] = pd.to_datetime(df['time:timestamp'])

# Compute durations for each case
case_durations = (
    df.groupby('case:concept:name')['time:timestamp']
    .agg(['min', 'max'])
    .assign(duration_hours=lambda x: (x['max'] - x['min']).dt.total_seconds() / 3600)
)

# Summary statistics
mean_dur = case_durations['duration_hours'].mean()
std_dur = case_durations['duration_hours'].std()
median_dur = case_durations['duration_hours'].median()

# Histogram
plt.figure(figsize=(12, 6))
sns.histplot(case_durations['duration_hours'], bins=30, kde=True, color="skyblue")
plt.axvline(mean_dur, color='red', linestyle='dashed', linewidth=2, label=f"Mean: {{mean_dur:.2f}}h")
plt.axvline(mean_dur + std_dur, color='orange', linestyle='dotted', linewidth=2, label=f"Mean + 1 STD: {{mean_dur + std_dur:.2f}}h")
plt.axvline(mean_dur - std_dur, color='orange', linestyle='dotted', linewidth=2, label=f"Mean - 1 STD: {{mean_dur - std_dur:.2f}}h")
plt.axvline(median_dur, color='green', linestyle='dashdot', linewidth=2, label=f"Median: {{median_dur:.2f}}h")
plt.title("Case Duration Distribution")
plt.xlabel("Duration (hours)")
plt.ylabel("Number of Cases")
plt.legend()
plt.tight_layout()
plt.show()

# Print summary stats
print(f"Mean: {{mean_dur:.2f}}h | Std Dev: {{std_dur:.2f}}h | Median: {{median_dur:.2f}}h")
print(f"Min: {{case_durations['duration_hours'].min():.2f}}h | Max: {{case_durations['duration_hours'].max():.2f}}h")


---

Question: Generate a bar chart of activity frequencies, sorted to emphasize the most and least frequent activities

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Count activity frequencies
activity_counts = df['concept:name'].value_counts().reset_index()
activity_counts.columns = ['Activity', 'Frequency']

# Sort from most to least frequent
activity_counts = activity_counts.sort_values(by='Frequency', ascending=False)

# Create bar chart
plt.figure(figsize=(12, 6))
sns.barplot(x='Activity', y='Frequency', data=activity_counts)
plt.xticks(rotation=45, ha='right')
plt.title('Frequency of Activities')
plt.xlabel('Activity')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


---

Question: Create a heatmap showing the number of events executed each day of the week across all resources.

```python
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Garantir que o timestamp está em datetime
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'])

# Extrair dia da semana e nome do recurso
df['day_of_week'] = df['time:timestamp'].dt.day_name()
df['resource'] = df['org:resource']

# Criar tabela cruzada: dias da semana × recursos
heatmap_data = df.pivot_table(
    index='day_of_week',
    columns='resource',
    aggfunc='size',
    fill_value=0
)

# Reordenar dias da semana
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = heatmap_data.reindex(days_order)

# Plotar heatmap
plt.figure(figsize=(14, 6))
sns.heatmap(heatmap_data, annot=True, fmt="d", cmap="YlGnBu", cbar=True)
plt.title("Number of Events by Resource and Day of the Week")
plt.xlabel("Resource")
plt.ylabel("Day of the Week")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


Now, given the following question, generate the code:

Question: "{question}"

"""
,

"FIVE_SHOT": """You are a process mining assistant that analyzes event logs and generates Python code to create a visualization in response to a process analysis question.

The log is loaded in the variable `log` (PM4PY EventLog object), and a corresponding pandas DataFrame is available as `df`. 

**Relevant Columns:** Specify which columns are necessary to answer the question and why (e.g., `concept:name`, `case:concept:name`, `time:timestamp`, `org:group`, etc.).

Your task is to return only executable Python code that produces the appropriate plot to answer the question. Do not include explanations or comments.

---

Question: Show a Directly-Follows Graph that highlights how activities are connected in the process, based on frequency.

```python
from pm4py.algo.discovery.dfg import algorithm as dfg_discovery
from pm4py.visualization.dfg import visualizer as dfg_visualization
from IPython.display import Image, display

# Generate DFG from the event log
dfg = dfg_discovery.apply(log)

# Configure visualization parameters
parameters = {{
    "format": "png",
    "bgcolor": "white",
    "edge_threshold": 0.01
}}

# Save and display the DFG visualization
gviz = dfg_visualization.apply(dfg, log=log, parameters=parameters)
gviz


---

Question: Create a histogram showing how long cases typically last in the process, with mean and standard deviation highlighted.

```python
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Convert timestamp to datetime
if not pd.api.types.is_datetime64_any_dtype(df['time:timestamp']):
    df['time:timestamp'] = pd.to_datetime(df['time:timestamp'])

# Compute durations for each case
case_durations = (
    df.groupby('case:concept:name')['time:timestamp']
    .agg(['min', 'max'])
    .assign(duration_hours=lambda x: (x['max'] - x['min']).dt.total_seconds() / 3600)
)

# Summary statistics
mean_dur = case_durations['duration_hours'].mean()
std_dur = case_durations['duration_hours'].std()
median_dur = case_durations['duration_hours'].median()

# Histogram
plt.figure(figsize=(12, 6))
sns.histplot(case_durations['duration_hours'], bins=30, kde=True, color="skyblue")
plt.axvline(mean_dur, color='red', linestyle='dashed', linewidth=2, label=f"Mean: {{mean_dur:.2f}}h")
plt.axvline(mean_dur + std_dur, color='orange', linestyle='dotted', linewidth=2, label=f"Mean + 1 STD: {{mean_dur + std_dur:.2f}}h")
plt.axvline(mean_dur - std_dur, color='orange', linestyle='dotted', linewidth=2, label=f"Mean - 1 STD: {{mean_dur - std_dur:.2f}}h")
plt.axvline(median_dur, color='green', linestyle='dashdot', linewidth=2, label=f"Median: {{median_dur:.2f}}h")
plt.title("Case Duration Distribution")
plt.xlabel("Duration (hours)")
plt.ylabel("Number of Cases")
plt.legend()
plt.tight_layout()
plt.show()

# Print summary stats
print(f"Mean: {{mean_dur:.2f}}h | Std Dev: {{std_dur:.2f}}h | Median: {{median_dur:.2f}}h")
print(f"Min: {{case_durations['duration_hours'].min():.2f}}h | Max: {{case_durations['duration_hours'].max():.2f}}h")


---

Question: Generate a bar chart of activity frequencies, sorted to emphasize the most and least frequent activities

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Count activity frequencies
activity_counts = df['concept:name'].value_counts().reset_index()
activity_counts.columns = ['Activity', 'Frequency']

# Sort from most to least frequent
activity_counts = activity_counts.sort_values(by='Frequency', ascending=False)

# Create bar chart
plt.figure(figsize=(12, 6))
sns.barplot(x='Activity', y='Frequency', data=activity_counts)
plt.xticks(rotation=45, ha='right')
plt.title('Frequency of Activities')
plt.xlabel('Activity')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


---

Question: Create a heatmap showing the number of events executed each day of the week across all resources.

```python
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Garantir que o timestamp está em datetime
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'])

# Extrair dia da semana e nome do recurso
df['day_of_week'] = df['time:timestamp'].dt.day_name()
df['resource'] = df['org:resource']

# Criar tabela cruzada: dias da semana × recursos
heatmap_data = df.pivot_table(
    index='day_of_week',
    columns='resource',
    aggfunc='size',
    fill_value=0
)

# Reordenar dias da semana
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = heatmap_data.reindex(days_order)

# Plotar heatmap
plt.figure(figsize=(14, 6))
sns.heatmap(heatmap_data, annot=True, fmt="d", cmap="YlGnBu", cbar=True)
plt.title("Number of Events by Resource and Day of the Week")
plt.xlabel("Resource")
plt.ylabel("Day of the Week")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---

Question: Train a classifier using features derived from event logs and visualize the importance of the most relevant features in a bar chart

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Convert timestamp to datetime
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'])

# Compute duration per case
case_durations = (
    df.groupby('case:concept:name')['time:timestamp']
    .agg(['min', 'max'])
    .assign(duration=lambda x: (x['max'] - x['min']).dt.total_seconds())
    .reset_index()
)

# Create binary target: long cases (duration > average)
average_duration = case_durations['duration'].mean()
case_durations['target'] = (case_durations['duration'] > average_duration).astype(int)

# Extract first 3 activities per case
first_acts = (
    df.sort_values(['case:concept:name', 'time:timestamp'])
    .groupby('case:concept:name')
    .head(3)
    .groupby('case:concept:name')['concept:name']
    .apply(list)
    .reset_index()
)

# Keep only cases with at least 3 events
first_acts = first_acts[first_acts['concept:name'].apply(len) == 3]

# Merge features with target
data = pd.merge(first_acts, case_durations[['case:concept:name', 'target']], on='case:concept:name')

# Encode categorical activities
le = LabelEncoder()
X = pd.DataFrame({{
    'act1': le.fit_transform([x[0] for x in data['concept:name']]),
    'act2': le.fit_transform([x[1] for x in data['concept:name']]),
    'act3': le.fit_transform([x[2] for x in data['concept:name']])
}})
y = data['target']

# Train classifier
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

# Visualize feature importances
plt.figure(figsize=(8, 5))
sns.barplot(x=X.columns, y=clf.feature_importances_)
plt.title("Feature Importance from Event Log-Based Features")
plt.xlabel("Feature")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()


Now, given the following question, generate the code:

Question: "{question}"

""",


"SIX_SHOT": """You are a process mining assistant that analyzes event logs and generates Python code to create a visualization in response to a process analysis question.

The log is loaded in the variable `log` (PM4PY EventLog object), and a corresponding pandas DataFrame is available as `df`. 

1. **Relevant Columns:** Specify which columns are necessary to answer the question and why (e.g., `concept:name`, `case:concept:name`, `time:timestamp`, `org:group`, etc.).

Your task is to return only executable Python code that produces the appropriate plot to answer the question. Do not include explanations or comments.

---

Question: Show a Directly-Follows Graph that highlights how activities are connected in the process, based on frequency.

```python
from pm4py.algo.discovery.dfg import algorithm as dfg_discovery
from pm4py.visualization.dfg import visualizer as dfg_visualization
from IPython.display import Image, display

# Generate DFG from the event log
dfg = dfg_discovery.apply(log)

# Configure visualization parameters
parameters = {{
    "format": "png",
    "bgcolor": "white",
    "edge_threshold": 0.01
}}

# Save and display the DFG visualization
gviz = dfg_visualization.apply(dfg, log=log, parameters=parameters)
gviz


---

Question: Create a histogram showing how long cases typically last in the process, with mean and standard deviation highlighted.

```python
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Convert timestamp to datetime
if not pd.api.types.is_datetime64_any_dtype(df['time:timestamp']):
    df['time:timestamp'] = pd.to_datetime(df['time:timestamp'])

# Compute durations for each case
case_durations = (
    df.groupby('case:concept:name')['time:timestamp']
    .agg(['min', 'max'])
    .assign(duration_hours=lambda x: (x['max'] - x['min']).dt.total_seconds() / 3600)
)

# Summary statistics
mean_dur = case_durations['duration_hours'].mean()
std_dur = case_durations['duration_hours'].std()
median_dur = case_durations['duration_hours'].median()

# Histogram
plt.figure(figsize=(12, 6))
sns.histplot(case_durations['duration_hours'], bins=30, kde=True, color="skyblue")
plt.axvline(mean_dur, color='red', linestyle='dashed', linewidth=2, label=f"Mean: {{mean_dur:.2f}}h")
plt.axvline(mean_dur + std_dur, color='orange', linestyle='dotted', linewidth=2, label=f"Mean + 1 STD: {{mean_dur + std_dur:.2f}}h")
plt.axvline(mean_dur - std_dur, color='orange', linestyle='dotted', linewidth=2, label=f"Mean - 1 STD: {{mean_dur - std_dur:.2f}}h")
plt.axvline(median_dur, color='green', linestyle='dashdot', linewidth=2, label=f"Median: {{median_dur:.2f}}h")
plt.title("Case Duration Distribution")
plt.xlabel("Duration (hours)")
plt.ylabel("Number of Cases")
plt.legend()
plt.tight_layout()
plt.show()

# Print summary stats
print(f"Mean: {{mean_dur:.2f}}h | Std Dev: {{std_dur:.2f}}h | Median: {{median_dur:.2f}}h")
print(f"Min: {{case_durations['duration_hours'].min():.2f}}h | Max: {{case_durations['duration_hours'].max():.2f}}h")


---

Question: Generate a bar chart of activity frequencies, sorted to emphasize the most and least frequent activities

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Count activity frequencies
activity_counts = df['concept:name'].value_counts().reset_index()
activity_counts.columns = ['Activity', 'Frequency']

# Sort from most to least frequent
activity_counts = activity_counts.sort_values(by='Frequency', ascending=False)

# Create bar chart
plt.figure(figsize=(12, 6))
sns.barplot(x='Activity', y='Frequency', data=activity_counts)
plt.xticks(rotation=45, ha='right')
plt.title('Frequency of Activities')
plt.xlabel('Activity')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


---

Question: Create a heatmap showing the number of events executed each day of the week across all resources.

```python
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Garantir que o timestamp está em datetime
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'])

# Extrair dia da semana e nome do recurso
df['day_of_week'] = df['time:timestamp'].dt.day_name()
df['resource'] = df['org:resource']

# Criar tabela cruzada: dias da semana × recursos
heatmap_data = df.pivot_table(
    index='day_of_week',
    columns='resource',
    aggfunc='size',
    fill_value=0
)

# Reordenar dias da semana
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = heatmap_data.reindex(days_order)

# Plotar heatmap
plt.figure(figsize=(14, 6))
sns.heatmap(heatmap_data, annot=True, fmt="d", cmap="YlGnBu", cbar=True)
plt.title("Number of Events by Resource and Day of the Week")
plt.xlabel("Resource")
plt.ylabel("Day of the Week")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---

Question: Train a classifier using features derived from event logs and visualize the importance of the most relevant features in a bar chart

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Convert timestamp to datetime
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'])

# Compute duration per case
case_durations = (
    df.groupby('case:concept:name')['time:timestamp']
    .agg(['min', 'max'])
    .assign(duration=lambda x: (x['max'] - x['min']).dt.total_seconds())
    .reset_index()
)

# Create binary target: long cases (duration > average)
average_duration = case_durations['duration'].mean()
case_durations['target'] = (case_durations['duration'] > average_duration).astype(int)

# Extract first 3 activities per case
first_acts = (
    df.sort_values(['case:concept:name', 'time:timestamp'])
    .groupby('case:concept:name')
    .head(3)
    .groupby('case:concept:name')['concept:name']
    .apply(list)
    .reset_index()
)

# Keep only cases with at least 3 events
first_acts = first_acts[first_acts['concept:name'].apply(len) == 3]

# Merge features with target
data = pd.merge(first_acts, case_durations[['case:concept:name', 'target']], on='case:concept:name')

# Encode categorical activities
le = LabelEncoder()
X = pd.DataFrame({{
    'act1': le.fit_transform([x[0] for x in data['concept:name']]),
    'act2': le.fit_transform([x[1] for x in data['concept:name']]),
    'act3': le.fit_transform([x[2] for x in data['concept:name']])
}})
y = data['target']

# Train classifier
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

# Visualize feature importances
plt.figure(figsize=(8, 5))
sns.barplot(x=X.columns, y=clf.feature_importances_)
plt.title("Feature Importance from Event Log-Based Features")
plt.xlabel("Feature")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

---

Question: Simulate the impact of delaying a specific activity by a fixed amount of time in all cases, and visualize how this affects the overall distribution of case durations.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Convert timestamps
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'])

# Define activity to simulate delay
activity_to_delay = 'Approve'
delay_minutes = 60  # 1 hour

# Create a copy of the dataframe
df_simulated = df.copy()

# Apply delay to the selected activity
mask = df_simulated['concept:name'] == activity_to_delay
df_simulated.loc[mask, 'time:timestamp'] += pd.Timedelta(minutes=delay_minutes)

# Compute original case durations
original_durations = (
    df.groupby('case:concept:name')['time:timestamp']
    .agg(['min', 'max'])
    .assign(duration=lambda x: (x['max'] - x['min']).dt.total_seconds() / 3600)
    .rename(columns={{'duration': 'Original Duration'}})
)

# Compute simulated case durations
simulated_durations = (
    df_simulated.groupby('case:concept:name')['time:timestamp']
    .agg(['min', 'max'])
    .assign(duration=lambda x: (x['max'] - x['min']).dt.total_seconds() / 3600)
    .rename(columns={{'duration': 'Simulated Duration'}})
)

# Combine for comparison
durations_comparison = original_durations.join(simulated_durations['Simulated Duration'])

# Plot histograms
plt.figure(figsize=(12, 6))
sns.histplot(durations_comparison['Original Duration'], color='blue', label='Original', bins=30, kde=True)
sns.histplot(durations_comparison['Simulated Duration'], color='orange', label='With Delay', bins=30, kde=True)
plt.title(f"Effect of Delaying '{{activity_to_delay}}' by {{delay_minutes}} Minutes")
plt.xlabel("Case Duration (hours)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()


Now, given the following question, generate the code:

Question: "{question}"

"""


,
"CHAIN_OF_THOUGHT": """You are a process mining assistant that analyzes event logs to answer questions related to processes.

You are provided with an event log stored in the variable `log` (PM4PY EventLog object), and a corresponding DataFrame `df` (pandas) already extracted from the log. Use PM4PY for process discovery and pandas, matplotlib, or seaborn when appropriate for data manipulation and visualization.

1. **Relevant Columns:** Specify which columns are necessary to answer the question and why (e.g., `concept:name`, `case:concept:name`, `time:timestamp`, `org:group`, etc.).

Your task is to **reason step by step** and explain how the answer should be derived using the event log data, using relevant functionality from PM4PY, pandas, or matplotlib/seaborn.

Structure your response as follows:

1. **Relevant Columns:** Identify which columns are necessary and why (e.g., activity, timestamp, case ID, resource).

2. **Processing Steps:** Describe how the data should be manipulated:
   - How to filter, group, or transform it;
   - What to compute (e.g., durations, frequencies, directly-follows, distributions, correlations);
   - What comparisons or aggregations are needed.

3. **Visualization:** Describe the best chart type to represent the results (e.g., bar chart, box plot, histogram, heatmap, DFG, Sankey). Specify axes, annotations, and formatting if relevant.

4. **Expected Insight:** State what the plot is expected to reveal — trends, bottlenecks, variations, anomalies, or structure.

Finally, return only the executable Python code that performs the task. The code must be clean, correct, and robust. If needed, include `print` statements to show intermediate results. Use best practices for clarity and reproducibility.

Here is a generic case example from an event log:

Generic Case Example:
At 2024-01-01 08:00:00, activity 'Activity A' was performed by Resource X.
At 2024-01-01 08:15:00, activity 'Activity B' was performed by Resource Y.
At 2024-01-01 08:30:00, activity 'Activity C' was performed by Resource X.
At 2024-01-01 08:45:00, activity 'Activity D' was performed by Resource Z.
At 2024-01-01 09:00:00, activity 'Activity E' was performed by Resource Y.
At 2024-01-01 09:30:00, activity 'Activity F' was performed by Resource W.
At 2024-01-01 10:00:00, activity 'Activity G' was performed by Resource X.
At 2024-01-01 10:30:00, activity 'Activity H' was performed by Resource V.
At 2024-01-01 11:00:00, activity 'Activity I' was performed by Resource Y.
At 2024-01-01 11:45:00, activity 'Activity J' was performed by Resource Z.
At 2024-01-01 12:00:00, activity 'Activity K' was performed by Resource W.
At 2024-01-01 12:30:00, activity 'Activity L' was performed by Resource V.
At 2024-01-01 13:00:00, activity 'Activity M' was performed by Resource X.
At 2024-01-01 13:30:00, activity 'Activity N' was performed by Resource Y.
At 2024-01-01 14:00:00, activity 'Activity O' was performed by Resource Z.
At 2024-01-01 14:30:00, activity 'Activity P' was performed by Resource W.
At 2024-01-01 15:00:00, activity 'Activity Q' was performed by Resource X.
At 2024-01-01 15:30:00, activity 'Activity R' was performed by Resource Y.
At 2024-01-01 16:00:00, activity 'Activity S' was performed by Resource Z.
At 2024-01-01 16:30:00, activity 'Activity T' was performed by Resource V.
At 2024-01-01 17:00:00, activity 'Activity U' was performed by Resource W.
At 2024-01-01 17:30:00, activity 'Activity V' was performed by Resource X.
At 2024-01-01 18:00:00, activity 'Activity W' was performed by Resource Y.
At 2024-01-01 18:30:00, activity 'Activity X' was performed by Resource Z.
At 2024-01-01 19:00:00, activity 'Activity Y' was performed by Resource W.
At 2024-01-01 19:30:00, activity 'Activity Z' was performed by Resource V.

Question:
"{question}"
"""
}

### 5.1.2 Assessment

In [ ]:
EVALUATION_PROMPTS = {
    "evaluation_code": """
Given the following question:
\"\"\"{question}\"\"\"

And the following generated code and its output:
\"\"\"Code:
{generated_code}

Output:
{output_text}
\"\"\"

How would you grade the above answer on a scale from 1.0 (minimum) to 10.0 (maximum)? 
Please respond **only** with a single numeric value (e.g., 7.5).
"""
}

# 6. Initialize Agent

In [ ]:
agent_sepsis = ProcessMiningAgent(
    generation_prompts=GENERATION_PROMPTS,
    evaluation_prompts=EVALUATION_PROMPTS,
    api_keys=API_KEYS
)

agent_eletronic_invoice = ProcessMiningAgent(
    generation_prompts=GENERATION_PROMPTS,
    evaluation_prompts=EVALUATION_PROMPTS,
    api_keys=API_KEYS
)
agent_request_for_payment = ProcessMiningAgent(
    generation_prompts=GENERATION_PROMPTS,
    evaluation_prompts=EVALUATION_PROMPTS,
    api_keys=API_KEYS
)

# 7. Configurations 

## 7.1 Sepsis

In [ ]:
models = [
    {"model_name": "gpt-4.1-2025-04-14", "model_type": "openai"},
]

dataset_names = [
    "sepsis",
    "EletronicInvoice",
    "RequestForPayment"
]

prompts = [
    "ZERO_SHOT",
    "FOUR_SHOT",
    "SIX_SHOT",
    "CHAIN_OF_THOUGHT"
]

test_configs_all = []

for dataset in dataset_names:
    for model in models:
        for prompt in prompts:
            config = {
                "nome": f"{dataset}-{model['model_name']}-{prompt}",
                "dataset": dataset,
                "gen_model_type": model["model_type"],
                "gen_model_name": model["model_name"],
                "eval_model_type": "gemini",
                "eval_model_name": "gemini-2.5-flash-preview-04-17",
                "prompt_name": prompt,
                "eval_prompt_name": "evaluation_code"
            }
            test_configs_all.append(config)

test_configs_by_dataset = {
    dataset: [cfg for cfg in test_configs_all if cfg["dataset"] == dataset]
    for dataset in dataset_names
}

In [ ]:
for dataset, configs in test_configs_by_dataset.items():
    print(f"\nDataset: {dataset}")
    for config in configs:
        print(f"  - {config['nome']}")
        print(f"      Gen Model: {config['gen_model_type']} - {config['gen_model_name']}")
        print(f"      Eval Model: {config['eval_model_type']} - {config['eval_model_name']}")
        print(f"      Prompt: {config['prompt_name']}")

# 8. Execution

In [ ]:
agents = {
    "sepsis": agent_sepsis,
    "EletronicInvoice": agent_eletronic_invoice,
    "RequestForPayment": agent_request_for_payment
}

In [ ]:
NUM_ITERATIONS = 5
BATCH_SIZE = 7

overall_results = {}

for ds_key, configs in test_configs_by_dataset.items():
    agent = agents[ds_key]
    dataset = datasets[ds_key]

    for config in configs:
        config_name = config["nome"]
        overall_results[config_name] = {}

        print(f"\n=== Executando {config_name} por {NUM_ITERATIONS} iterações ===")
        for run in range(1, NUM_ITERATIONS + 1):
            print(f"\n--- Iteração {run}/{NUM_ITERATIONS} ---")
            run_output_root = os.path.join(output_root, f"{config_name.replace(':', '_')}_run{run}")

            questions = dataset['questions']
            num_batches = len(questions) // BATCH_SIZE + (1 if len(questions) % BATCH_SIZE != 0 else 0)

            for batch_idx in range(num_batches):
                batch_df = questions.iloc[batch_idx * BATCH_SIZE : (batch_idx + 1) * BATCH_SIZE]
                print(f"    >> Batch {batch_idx + 1}/{num_batches}")

                batch_output_root = os.path.join(run_output_root, f"batch_{batch_idx+1}")

                all_done = True
                for i in range(len(batch_df)):
                    question_folder = os.path.join(batch_output_root, f"Question_{i+1}")
                    if not os.path.exists(question_folder):
                        all_done = False
                        break
                if all_done:
                    print(f"    >> Batch {batch_idx + 1} já processado. Pulando.")
                    continue

                results = agent.run_process_mining_batch(
                    questions_df=batch_df,
                    output_root=batch_output_root,
                    gen_model_type=config['gen_model_type'],
                    gen_model_name=config['gen_model_name'],
                    eval_model_type=config['eval_model_type'],
                    eval_model_name=config['eval_model_name'],
                    prompt_name=config['prompt_name'],
                    eval_prompt_name=config['eval_prompt_name'],
                    log=dataset['log'],
                    df=dataset['df'],
                    pause_between_questions=False
                )

                for r in results:
                    q = r['question']
                    nota = r['nota_final']
                    overall_results[config_name].setdefault(q, []).append(nota)

rows = []
for config_name, questions_dict in overall_results.items():
    question_to_id = {q: idx+1 for idx, q in enumerate(reversed(questions_dict.keys()))} 

    for question, notas in questions_dict.items():
        row = {
            'ID': question_to_id[question],
            'Configuração': config_name,
            'Pergunta': question,
            **{f'Run {i+1}': nota for i, nota in enumerate(notas)},
            'Média': sum(notas) / len(notas)
        }
        rows.append(row)

df_summary_o4_mini = pd.DataFrame(rows)

df_summary_o4_mini[['Dataset', 'Modelo', 'Prompt']] = df_summary_o4_mini['Configuração'].str.extract(r'^(.*?)-(.*?)-(.*)$')

def get_categoria(id_value):
    if 1 <= id_value <= 5:
        return 'Q1'
    elif 6 <= id_value <= 8:
        return 'Q2'
    elif 9 <= id_value <= 11:
        return 'Q3'
    elif 12 <= id_value <= 15:
        return 'Q4'
    elif 16 <= id_value <= 19:
        return 'Q5'
    elif 20 <= id_value <= 23:
        return 'Q7'
    elif 24 <= id_value <= 25:
        return 'Q8'
    elif 26 <= id_value <= 28:
        return 'Q9'
    else:
        return 'N/A'

df_summary_o4_mini['Categoria'] = df_summary_o4_mini['ID'].apply(get_categoria)

run_cols = [col for col in df_summary_o4_mini.columns if col.startswith('Run ')]
final_cols = ['ID', 'Categoria', 'Dataset', 'Modelo', 'Prompt', 'Configuração', 'Pergunta'] + run_cols + ['Média']
df_summary_o4_mini = df_summary_o4_mini[final_cols]

print(df_summary_o4_mini)
summary_file = os.path.join(output_root, 'summary.xlsx')
df_summary_o4_mini.to_excel(summary_file, index=False)
print(f"\nResumo exportado para: {summary_file}")